In [106]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score,make_scorer,f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.model_selection import train_test_split,RandomizedSearchCV,cross_val_score
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE,ADASYN
from xgboost import  XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from collections import Counter
import warnings
import joblib
import optuna

In [107]:
warnings.filterwarnings('ignore')
model_dt=DecisionTreeClassifier()
model_dt1=DecisionTreeClassifier()
model_dt2=DecisionTreeClassifier()
model_dt_smeen=DecisionTreeClassifier()
model_rf_smenn=RandomForestClassifier(n_estimators=500)
model_rf1=RandomForestClassifier(class_weight='balanced',n_estimators=300,max_depth=6,random_state=42)
model_xbg=XGBClassifier(random_state=42)
model_xgb_smote=XGBClassifier(random_state=42)
model_xgb_smenn=XGBClassifier(random_state=42)
model_xgb_adasyn=XGBClassifier(random_state=42)
model_xgb_randomizeds=XGBClassifier(random_state=42,eval_metric='logloss')
model_lgb=LGBMClassifier(random_state=42)
model_catb=CatBoostClassifier(auto_class_weights='Balanced',verbose=0,random_state=42)
model_ada_weighted=AdaBoostClassifier(random_state=42,n_estimators=50)
sc=StandardScaler()
mms=MinMaxScaler()
smenn=SMOTEENN()
smote=SMOTE(random_state=42)
adasyn=ADASYN(random_state=42)


In [108]:
df=pd.read_csv('../data/Customer-Churn.csv')

In [109]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [110]:
df.Churn.value_counts()/len(df)*100

Churn
No     73.463013
Yes    26.536987
Name: count, dtype: float64

## **Churn Rate**: 26.53%

- Which means, 26.53% of the customers churn out of this telecom company

In [111]:
y=df['Churn']

x=df.drop(columns=['customerID','Churn'])


print("Shape of x is : ",x.shape)
print("Shape of y is : ",y.shape)

Shape of x is :  (7043, 19)
Shape of y is :  (7043,)


## Train Test Split

In [112]:
x=pd.get_dummies(x,drop_first=True)
y=df['Churn'].map({'No':0,'Yes':1})

In [113]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

## Model Building

In [114]:
model_dt=DecisionTreeClassifier()
model_dt.fit(x_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [115]:
y_pred_dt=model_dt.predict(x_test)

In [116]:
print("Classification Report : ")
print(classification_report(y_test,y_pred_dt))

Classification Report : 
              precision    recall  f1-score   support

           0       0.81      0.87      0.84      1009
           1       0.59      0.49      0.53       400

    accuracy                           0.76      1409
   macro avg       0.70      0.68      0.69      1409
weighted avg       0.75      0.76      0.75      1409



In [117]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Initial Insights

- Base model has a accuracy of 76% which is not reliable because of the imbalanced dataset
- TotalCharges needs to be a float/int type (Data Cleaning)
- Need to perform Feature Scaling

## Data Cleaning

In [118]:
telco_df=df.copy()

In [119]:
telco_df.TotalCharges=pd.to_numeric(telco_df.TotalCharges,errors='coerce')

In [120]:
telco_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [121]:
telco_df.loc[telco_df['TotalCharges'].isnull()==True]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [122]:
telco_df.dropna(how='any',inplace=True)

In [123]:
telco_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [124]:
telco_df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [125]:
labels=['1-12','13-24','25-36','37-48','49-60',
        61-72]
bins=[0,12,24,36,48,60,72]
telco_df['tenure_group']=pd.cut(x=telco_df.tenure,labels=labels,bins=bins,right=True,include_lowest=True)

print(telco_df[['tenure','tenure_group']].head(10))

print(telco_df['tenure_group'].value_counts().sort_index())

   tenure tenure_group
0       1         1-12
1      34        25-36
2       2         1-12
3      45        37-48
4       2         1-12
5       8         1-12
6      22        13-24
7      10         1-12
8      28        25-36
9      62          -11
tenure_group
1-12     2175
13-24    1024
25-36     832
37-48     762
49-60     832
-11      1407
Name: count, dtype: int64


In [126]:
telco_df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1-12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-36
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1-12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37-48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1-12


In [127]:
y=telco_df['Churn']
x=telco_df.drop(columns=['Churn','customerID','tenure'])

print("shape of x is : ",x.shape)
print("shape of y is :",y.shape)

shape of x is :  (7032, 19)
shape of y is : (7032,)


In [128]:
x

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,tenure_group
0,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,1-12
1,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,25-36
2,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1-12
3,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,37-48
4,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,13-24
7039,Female,0,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,-11
7040,Female,0,Yes,Yes,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,1-12
7041,Male,1,Yes,No,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1-12


In [129]:
x=pd.get_dummies(x,drop_first=True)
y=telco_df['Churn'].map({'No':0,'Yes':1})

In [130]:
x

,SeniorCitizen,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_13-24,tenure_group_25-36,tenure_group_37-48,tenure_group_49-60,tenure_group_-11
0,0,29.85,29.85,False,True,False,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
1,0,56.95,1889.50,True,False,False,True,False,False,False,...,False,False,False,False,True,False,True,False,False,False
2,0,53.85,108.15,True,False,False,True,False,False,False,...,False,True,False,False,True,False,False,False,False,False
3,0,42.30,1840.75,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,False,False,False,True,False,False,True,...,False,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,84.80,1990.50,True,True,True,True,False,True,False,...,False,True,False,False,True,True,False,False,False,False
7039,0,103.20,7362.90,False,True,True,True,False,True,True,...,False,True,True,False,False,False,False,False,False,True
7040,0,29.60,346.45,False,True,True,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
7041,1,74.40,306.60,True,True,False,True,False,True,True,...,False,True,False,False,True,False,False,False,False,False


In [131]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

## Feature Scaling

In [132]:
x_train_sc=sc.fit_transform(x_train)
x_test_sc=sc.transform(x_test)

In [133]:
model_dt1.fit(x_train_sc,y_train)
y_pred_dt1=model_dt1.predict(x_test_sc)
print("Classification Report : \n",classification_report(y_test,y_pred_dt1))

Classification Report : 
               precision    recall  f1-score   support

           0       0.81      0.82      0.81      1033
           1       0.48      0.47      0.47       374

    accuracy                           0.72      1407
   macro avg       0.65      0.64      0.64      1407
weighted avg       0.72      0.72      0.72      1407



## Feature Scaling - MinMaxScaler

In [134]:
x_train_mms=mms.fit_transform(x_train)
x_test_mms=mms.transform(x_test)

In [135]:
model_dt2.fit(x_train_mms,y_train)
y_pred_dt2=model_dt2.predict(x_test)
print("Classification Report is : \n",classification_report(y_test,y_pred_dt2))

Classification Report is : 
               precision    recall  f1-score   support

           0       0.74      0.88      0.80      1033
           1       0.28      0.13      0.18       374

    accuracy                           0.68      1407
   macro avg       0.51      0.50      0.49      1407
weighted avg       0.61      0.68      0.63      1407



## SMOTEENN () [UpSampling + ENN)

In [136]:
x_train_resampled,y_train_resampled=smenn.fit_resample(x_train,y_train)

In [137]:
model_dt_smeen.fit(x_train_resampled,y_train_resampled)
y_pred_dt2_smeen=model_dt_smeen.predict(x_test)

print("Classification Report is :",classification_report(y_test,y_pred_dt2_smeen))

Classification Report is :               precision    recall  f1-score   support

           0       0.86      0.73      0.79      1033
           1       0.48      0.67      0.56       374

    accuracy                           0.72      1407
   macro avg       0.67      0.70      0.68      1407
weighted avg       0.76      0.72      0.73      1407



### Random Forest with Smotenn

In [138]:
model_rf_smenn.fit(x_train_resampled,y_train_resampled)
y_pred_rf_smenn=model_rf_smenn.predict(x_test)

In [139]:
print("Classification Report : \n",classification_report(y_test,y_pred_rf_smenn))

Classification Report : 
               precision    recall  f1-score   support

           0       0.88      0.77      0.82      1033
           1       0.53      0.72      0.61       374

    accuracy                           0.76      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.79      0.76      0.77      1407



### Xgbbost without SMOTEENN

In [140]:
model_xbg.fit(x_train,y_train)
Y_pred_xgb=model_xbg.predict(x_test)

print("Classification Report is : \n",classification_report(y_test,Y_pred_xgb))

Classification Report is : 
               precision    recall  f1-score   support

           0       0.83      0.88      0.86      1033
           1       0.61      0.51      0.55       374

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.78      1407



### Xgbbost with SMOTEENN

In [141]:
model_xgb_smenn.fit(x_train_resampled,y_train_resampled)
y_pred_xgb_smenn=model_xgb_smenn.predict(x_test)
print("Classification Report is :\n",classification_report(y_test,y_pred_xgb_smenn))

Classification Report is :
               precision    recall  f1-score   support

           0       0.88      0.78      0.82      1033
           1       0.53      0.70      0.60       374

    accuracy                           0.76      1407
   macro avg       0.70      0.74      0.71      1407
weighted avg       0.78      0.76      0.76      1407



### XGBoost with SMOTE

In [142]:
print("Before Smote :",Counter(y_train))

x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)

print("After Smote : ",Counter(y_train_smote))

model_xgb_smote.fit(x_train_smote,y_train_smote)

y_pred_xgb_smote=model_xgb_smote.predict(x_test)

print("classification report is : ",classification_report(y_test,y_pred_xgb_smote))


Before Smote : Counter({0: 4130, 1: 1495})
After Smote :  Counter({1: 4130, 0: 4130})
classification report is :                precision    recall  f1-score   support

           0       0.84      0.84      0.84      1033
           1       0.55      0.54      0.55       374

    accuracy                           0.76      1407
   macro avg       0.69      0.69      0.69      1407
weighted avg       0.76      0.76      0.76      1407



### Xgboost with Adaysn

In [143]:
print("Before Adaysn : ",Counter(y_train))

x_train_adaysn,y_train_adaysn=adasyn.fit_resample(x_train,y_train)

print("After Adasyn : ",Counter(y_train_adaysn))

model_xgb_adasyn.fit(x_train_adaysn,y_train_adaysn)

y_pred_xgb_adasyn=model_xgb_adasyn.predict(x_test)

print("Accuracy is : ",accuracy_score(y_test,y_pred_xgb_adasyn))
print("Classification Report is : ",classification_report(y_test,y_pred_xgb_adasyn))
print("confusion matrix is : \n",confusion_matrix(y_test,y_pred_xgb_adasyn))

Before Adaysn :  Counter({0: 4130, 1: 1495})
After Adasyn :  Counter({1: 4187, 0: 4130})
Accuracy is :  0.7661691542288557
Classification Report is :                precision    recall  f1-score   support

           0       0.84      0.85      0.84      1033
           1       0.56      0.55      0.55       374

    accuracy                           0.77      1407
   macro avg       0.70      0.70      0.70      1407
weighted avg       0.76      0.77      0.77      1407

confusion matrix is : 
 [[874 159]
 [170 204]]


In [144]:
scale_pos_weight=(y_train==0).sum()/(y_train==1).sum()
print("Scale pos Weight is : ",scale_pos_weight)

model_xgb_weighted=XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

model_xgb_weighted.fit(x_train,y_train)
y_pred_xgb_weighted=model_xgb_weighted.predict(x_test)

print("Accuracy score is : ",accuracy_score(y_test,y_pred_xgb_weighted))
print("classification_report is :\n ",classification_report(y_test,y_pred_xgb_weighted))
print("confusion_matrix is :\n",confusion_matrix(y_test,y_pred_xgb_weighted))

Scale pos Weight is :  2.762541806020067
Accuracy score is :  0.7668798862828714
classification_report is :
                precision    recall  f1-score   support

           0       0.87      0.80      0.83      1033
           1       0.55      0.68      0.61       374

    accuracy                           0.77      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.79      0.77      0.77      1407

confusion_matrix is :
 [[823 210]
 [118 256]]


### Adaboost with Weighted sample Weight

In [145]:
negative_count=(y_train==0).sum()
positive_count=(y_train==1).sum()
weighted_positive=negative_count/positive_count
print("Negative Weight is : ",negative_count)
print("Postive Count is : ",positive_count)
print("weighted is :",weighted_positive)

sample_weight=np.where(y_train==1,weighted_positive,1.0)

model_ada_weighted.fit(x_train,y_train,sample_weight=sample_weight)

y_pred_ada_weighted=model_ada_weighted.predict(x_test)

print("\nAdaboost with Sample Weighting accuracy is :\n",accuracy_score(y_test,y_pred_ada_weighted))
print("\n Classification Report \n",classification_report(y_test,y_pred_ada_weighted))
print("\nConfusion Matrix is : \n",confusion_matrix(y_test,y_pred_ada_weighted))

Negative Weight is :  4130
Postive Count is :  1495
weighted is : 2.762541806020067

Adaboost with Sample Weighting accuracy is :
 0.7455579246624022

 Classification Report 
               precision    recall  f1-score   support

           0       0.90      0.74      0.81      1033
           1       0.51      0.77      0.62       374

    accuracy                           0.75      1407
   macro avg       0.71      0.75      0.71      1407
weighted avg       0.80      0.75      0.76      1407


Confusion Matrix is : 
 [[761 272]
 [ 86 288]]


## **Hyper Parameter Optimization**

In [146]:
param_grid={
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 400],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight]
}
xgb=XGBClassifier(
    random_state=42,eval_metric='logloss'
)

search=RandomizedSearchCV(
                            xgb,
                            param_grid,
                            n_iter=50,
                            cv=5,
                            n_jobs=-1,
                            scoring='f1',
                            random_state=42
                              )
search.fit(x_train,y_train)

print("best Parameters :\n",search.best_params_)
print("Best CV F11 score is :",search.best_score_)

best_model=search.best_estimator_
y_pred_randomized=best_model.predict(x_test)
print("\n Test Classification Report is : \n",classification_report(y_test,y_pred_randomized))

best Parameters :
 {'subsample': 0.8, 'scale_pos_weight': np.float64(2.762541806020067), 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.7}
Best CV F11 score is : 0.6340280623972054

 Test Classification Report is : 
               precision    recall  f1-score   support

           0       0.91      0.72      0.81      1033
           1       0.51      0.80      0.63       374

    accuracy                           0.74      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.74      0.76      1407



In [147]:
joblib.dump(search,"../models/best_xgboost_churn_model.pkl")
print("Best model saved succesfully saved as best_xgboost_churn_model.pkl")

Best model saved succesfully saved as best_xgboost_churn_model.pkl


### Trying out few more techniques

In [148]:
model_rf1.fit(x_train,y_train)
y_pred_rf1=model_rf1.predict(x_test)
print("Classification Report is :\n",classification_report(y_test,y_pred_rf1))

Classification Report is :
               precision    recall  f1-score   support

           0       0.90      0.70      0.79      1033
           1       0.49      0.79      0.60       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.72      0.74      1407



### LightGBM Model

In [149]:
model_lgb.fit(x_train,y_train)
y_pred_lgb=model_lgb.predict(x_test)
print("classification report is : \n",classification_report(y_test,y_pred_lgb))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000166 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 574
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
classification report is : 
               precision    recall  f1-score   support

           0       0.82      0.89      0.86      1033
           1       0.61      0.48      0.54       374

    accuracy                           0.78      1407
   macro avg       0.72      0.68      0.70      1407
weighted avg       0.77      0.78      0.77      1407



### caatboost Model

In [150]:
model_catb.fit(x_train,y_train)
y_pred_catb=model_catb.predict(x_test)

print("Classification Report is \n",classification_report(y_test,y_pred_catb))

Classification Report is 
               precision    recall  f1-score   support

           0       0.90      0.77      0.83      1033
           1       0.54      0.75      0.63       374

    accuracy                           0.77      1407
   macro avg       0.72      0.76      0.73      1407
weighted avg       0.80      0.77      0.78      1407



## **Optuna fine tuning**

In [151]:
def objective(trial):
    params={
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),  
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2), 
        'scale_pos_weight': trial.suggest_categorical('scale_pos_weight', [1, scale_pos_weight]),
        'random_state': 42,
        'eval_metric': 'logloss'

    }

    model=XGBClassifier(**params)

    f1_scorer=make_scorer(f1_score,average='binary',pos_label=1)
    score=cross_val_score(model,x_train,y_train,cv=5,scoring=f1_scorer,n_jobs=-1).mean()

    return score
study=optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=100)


print("Best parameters ",study.best_params)
print("best cv f1 score : ",study.best_value)


best_model_optuna=XGBClassifier(**study.best_params)
best_model_optuna.fit(x_train,y_train)
y_pred_optuna=best_model_optuna.predict(x_test)
print("Classification Report is : ",classification_report(y_test,y_pred_optuna))

[I 2026-03-19 23:19:43,730] A new study created in memory with name: no-name-5c366fe3-eb17-4114-87dc-1f6ffd7dcdb1
[I 2026-03-19 23:19:44,224] Trial 0 finished with value: 0.5924023968114851 and parameters: {'max_depth': 4, 'learning_rate': 0.02015959308567891, 'n_estimators': 617, 'subsample': 0.6673942505378271, 'colsample_bytree': 0.8796657562525987, 'min_child_weight': 10, 'gamma': 0.47434495316208763, 'reg_alpha': 0.8993350537933548, 'reg_lambda': 1.4495143803462762, 'scale_pos_weight': 1}. Best is trial 0 with value: 0.5924023968114851.
[I 2026-03-19 23:19:44,903] Trial 1 finished with value: 0.538286664616256 and parameters: {'max_depth': 9, 'learning_rate': 0.10412280510660897, 'n_estimators': 576, 'subsample': 0.6507819168203691, 'colsample_bytree': 0.6873453088302044, 'min_child_weight': 1, 'gamma': 0.17340122352243031, 'reg_alpha': 0.43579321294601703, 'reg_lambda': 0.16381549640783266, 'scale_pos_weight': 1}. Best is trial 0 with value: 0.5924023968114851.
[I 2026-03-19 23:1

Best parameters  {'max_depth': 4, 'learning_rate': 0.021326169935095604, 'n_estimators': 280, 'subsample': 0.9592650404299282, 'colsample_bytree': 0.6719660122516147, 'min_child_weight': 4, 'gamma': 0.0038501484684124515, 'reg_alpha': 0.20335792547951695, 'reg_lambda': 1.4999513838437746, 'scale_pos_weight': np.float64(2.762541806020067)}
best cv f1 score :  0.6357847665144231
Classification Report is :                precision    recall  f1-score   support

           0       0.91      0.73      0.81      1033
           1       0.52      0.80      0.63       374

    accuracy                           0.75      1407
   macro avg       0.72      0.77      0.72      1407
weighted avg       0.81      0.75      0.76      1407



In [152]:
joblib.dump(best_model_optuna,"../models/best_optuna_churn_model.pkl")
print("Model is saved as best_optuna_model.pkl")

Model is saved as best_optuna_model.pkl


In [153]:
joblib.dump(model_ada_weighted,"../models/best_adaboost_churn_model.pkl")
print("Model is saved as best_adaboost.pkl")

Model is saved as best_adaboost.pkl
